# ROBUST04 Information Retrieval: Hybrid Multi-Stage Ranking with RRF Fusion

## Project Overview

This notebook implements a hybrid information retrieval system for the TREC ROBUST04 collection, combining multiple retrieval and reranking methods using **Reciprocal Rank Fusion (RRF)** as proposed by Cormack et al. (2009).

## Retrieval Methods Implemented

### 1. BM25 (Lexical Retrieval)
- **Rationale:** BM25 is a robust probabilistic retrieval model that excels at exact keyword matching. It serves as our primary first-stage retriever.
- **Parameters:** k1=0.9, b=0.4 (tuned for optimal performance with neural reranking)

### 2. RM3 (Pseudo-Relevance Feedback)
- **Rationale:** RM3 expands queries using terms from top-ranked documents, improving recall for queries with vocabulary mismatch.
- **Parameters:** fb_docs=15, fb_terms=120, original_query_weight=0.03
- **Finding:** Very low original_query_weight (0.03) works best with MonoT5-large, indicating the reranker benefits from aggressive query expansion.

### 3. MonoT5-large (Neural Reranker)
- **Rationale:** MonoT5 is a sequence-to-sequence reranker that provides deep semantic understanding of query-document relevance. The large variant (770M params) offers significant improvements over the base model.
- **Model:** castorini/monot5-large-msmarco-10k

### Methods Evaluated but Not Used

#### SPLADE (Learned Sparse Retrieval)
- **Tested:** Yes, as a 4th signal in RRF fusion
- **Result:** SPLADE actually **decreased** MAP from 0.3004 to 0.2937 (-2.2%)
- **Analysis:** The semantic signals from SPLADE overlap significantly with what BM25+RM3+MonoT5-large already capture. Adding it introduces noise rather than complementary information.

#### Contriever (Dense Retrieval)
- **Tested:** Yes, as a 4th signal in RRF fusion
- **Result:** Contriever **decreased** MAP from 0.2855 to 0.2805 (-1.8%)
- **Analysis:** Dense retrieval did not provide complementary signals for this task.

## Fusion Strategy: Reciprocal Rank Fusion (RRF)

Based on Cormack et al. (2009), we use the RRF formula:
```
RRFscore(d) = Σ 1/(k + rank(d))
```
- **k=10** (tuned; lower than the paper's k=60 for better performance with our setup)
- **Pure RRF:** All methods weighted equally (weight=1.0)
- **3-way fusion:** BM25 + RM3 + MonoT5-large

## Notable Findings

1. **Parameter Sensitivity:** BM25 parameters (k1, b) needed re-tuning after upgrading from MonoT5-base to MonoT5-large. Optimal shifted from (0.7, 0.6) to (0.9, 0.4).

2. **Query Expansion:** Aggressive RM3 expansion (original_query_weight=0.03) works best when paired with a strong neural reranker.

3. **Reranker Impact:** MonoT5-large provided ~8% MAP improvement over MonoT5-base, justifying the additional compute cost.

4. **RRF k-value:** k=10 outperformed the paper's recommended k=60 for our 3-way fusion setup.

5. **Less is More:** Adding more retrieval signals (SPLADE, Contriever) hurt performance. The 3-way combination proved optimal.

## Challenges Encountered

1. **Pyserini Version Compatibility:** Different pyserini versions have different APIs and prebuilt index availability, requiring careful package management.

2. **Compute Constraints:** MonoT5-large reranking takes ~130s per query on Colab T4 GPU, making full hyperparameter search time-intensive.

3. **Diminishing Returns from Additional Signals:** Both SPLADE and Contriever decreased performance when added to the pipeline, contrary to expectations.

## Final Performance

| Configuration | MAP | Change |
|--------------|-----|--------|
| BM25 baseline | 0.2531 | - |
| + RM3 + MonoT5-base (k_rrf=10) | 0.2855 | +12.8% |
| + MonoT5-large | 0.2923 | +15.5% |
| + Parameter tuning | **0.3004** | **+18.7%** |
| + SPLADE (4-way) | 0.2937 | -2.2% (rejected) |

## Final Configuration

- **BM25:** k1=0.9, b=0.4
- **RM3:** fb_docs=15, fb_terms=120, original_query_weight=0.03
- **Reranker:** MonoT5-large (castorini/monot5-large-msmarco-10k)
- **Fusion:** 3-way Pure RRF with k=10
- **Rerank depth:** 1000 documents

In [ ]:
hugging_face_token = "WRITE_YOUR_HF_TOKEN_HERE"

In [ ]:
from huggingface_hub import login
login(hugging_face_token)

In [ ]:
# ============================================================================
# JAVA 21 SETUP
# ============================================================================

import os
import subprocess

print("Checking Java version...")

try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

print("\n Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

print("\n Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 ready!")
print("="*70)

In [ ]:
# ============================================================================
# INSTALL PACKAGES
# ============================================================================

print("Installing packages...")

!pip install -q pyserini==0.22.1
!pip install -q transformers>=4.46.0
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers numpy
!pip install -q faiss-cpu --no-cache

print("\nAll packages installed!")

In [ ]:
import os
import torch
import numpy as np
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import warnings
import time

warnings.filterwarnings('ignore', message='Some weights of.*were not initialized')
warnings.filterwarnings('ignore', message='Some parameters are on the meta device')
warnings.filterwarnings('ignore', message='.*overflowing tokens are not returned.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP
# ============================================================================
from google.colab import drive
import os
import sys

try:
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("Google Drive mounted")

    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    if os.path.exists(BASE_DIR):
        print(f"Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
        print(f"Changed to: {os.getcwd()}")
    else:
        search_paths = [
            '/content/drive/MyDrive/TEXT/FinalProject',
            '/content/drive/MyDrive/FinalProject',
            '/content/drive/MyDrive/TEST/FinalProject',
            '/content/drive/MyDrive',
        ]

        for path in search_paths:
            query_file = os.path.join(path, 'queriesROBUST.txt')
            if os.path.exists(query_file):
                os.chdir(path)
                BASE_DIR = path
                break
else:
    BASE_DIR = os.getcwd()

print(f"\nChecking for required files...")
print(f"Current directory: {os.getcwd()}")

if os.path.exists('queriesROBUST.txt'):
    print(f"  Found: queriesROBUST.txt")
if os.path.exists('qrels_50_Queries'):
    print(f"  Found: qrels_50_Queries")

print("\n" + "="*70)
print("Setup complete!")
print("="*70)

## 1. Configuration

In [ ]:
# ============================================================================
# FINAL CONFIGURATION
# ============================================================================

# Model settings
USE_MONOT5 = True

# Optimized parameters (from extensive tuning)
BM25_K1 = 0.9
BM25_B = 0.4
RM3_FB_DOCS = 15
RM3_FB_TERMS = 120
RM3_ORIG_WEIGHT = 0.03
K_RRF = 10
RERANK_DEPTH = 1000

print("Final Configuration:")
print(f"  BM25: k1={BM25_K1}, b={BM25_B}")
print(f"  RM3: fb_docs={RM3_FB_DOCS}, fb_terms={RM3_FB_TERMS}, original_query_weight={RM3_ORIG_WEIGHT}")
print(f"  RRF: k={K_RRF}")
print(f"  Rerank depth: {RERANK_DEPTH}")
print(f"  Reranker: MonoT5-large")
print(f"\n3-Way Pure RRF: BM25 + RM3 + MonoT5-large")
print(f"\nExpected MAP: ~0.3004")

## 2. Load Index and Configure Searchers

In [ ]:
print("Loading ROBUST04 index...")

# Index reader for document retrieval
index_reader = IndexReader.from_prebuilt_index('robust04')

# BM25 searcher with OPTIMIZED parameters
bm25_searcher = LuceneSearcher.from_prebuilt_index('robust04')
bm25_searcher.set_bm25(k1=BM25_K1, b=BM25_B)

# RM3 searcher with OPTIMIZED parameters
rm3_searcher = LuceneSearcher.from_prebuilt_index('robust04')
rm3_searcher.set_bm25(k1=BM25_K1, b=BM25_B)
rm3_searcher.set_rm3(fb_docs=RM3_FB_DOCS, fb_terms=RM3_FB_TERMS, original_query_weight=RM3_ORIG_WEIGHT)

searchers = [bm25_searcher, rm3_searcher]

print("Index and searchers configured!")
print(f"  BM25: k1={BM25_K1}, b={BM25_B}")
print(f"  RM3: fb_docs={RM3_FB_DOCS}, fb_terms={RM3_FB_TERMS}, original_query_weight={RM3_ORIG_WEIGHT}")

## 3. Load Queries

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1]
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')
train_qids = sorted(qrels.keys())
test_qids = sorted([q for q in queries if q not in qrels])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

## 4. Helper Functions

In [ ]:
def classify_query(query_text):
    wc = len(query_text.split())
    return 'short' if wc <= 3 else 'medium' if wc <= 6 else 'long'

## 5. Multi-Method Retrieval

In [ ]:
def multi_method_retrieval(query_text, k=1000):
    """
    Get results from BM25 and RM3.
    """
    # BM25
    bm25_hits = searchers[0].search(query_text, k=k)
    bm25_results = [(h.docid, h.score) for h in bm25_hits]
    
    # RM3
    rm3_hits = searchers[1].search(query_text, k=k)
    rm3_results = [(h.docid, h.score) for h in rm3_hits]
    
    return bm25_results, rm3_results

## 6. Load MonoT5-large Reranker

In [ ]:
monot5_model, monot5_tokenizer = None, None
if USE_MONOT5:
    print("Loading MonoT5-large Reranker...")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    try:
        mn = "castorini/monot5-large-msmarco-10k"
        monot5_tokenizer = AutoTokenizer.from_pretrained(mn)
        monot5_model = AutoModelForSeq2SeqLM.from_pretrained(mn).to(device)
        monot5_model.eval()
        print(f"  MonoT5-large loaded successfully!")
        print(f"  Model: {mn}")
    except Exception as e:
        print(f"  MonoT5 load failed: {e}")
        USE_MONOT5 = False

## 7. MonoT5 Reranker Function

In [ ]:
def rerank_monot5(query, doc_ids, batch_size=16):
    """
    MonoT5 reranking (pointwise scoring)
    
    Returns dict of {doc_id: relevance_probability}
    Higher score = more relevant
    """
    if not USE_MONOT5 or monot5_model is None:
        return None

    doc_texts, valid_ids = [], []
    for did in doc_ids:
        try:
            doc = index_reader.doc(did)
            if doc:
                raw = doc.raw()
                if raw:
                    doc_texts.append(raw[:2000])
                    valid_ids.append(did)
        except Exception as e:
            pass

    if not doc_texts:
        return None

    prompts = [f"Query: {query} Document: {d} Relevant:" for d in doc_texts]

    true_token_id = monot5_tokenizer.encode("true")[0]
    false_token_id = monot5_tokenizer.encode("false")[0]

    all_scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        try:
            inputs = monot5_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            with torch.no_grad():
                outputs = monot5_model.generate(
                    **inputs,
                    max_new_tokens=1,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                logits = outputs.scores[0]
                true_logits = logits[:, true_token_id]
                false_logits = logits[:, false_token_id]
                probs = torch.softmax(torch.stack([false_logits, true_logits], dim=1), dim=1)
                batch_scores = probs[:, 1].cpu().tolist()
                all_scores.extend(batch_scores)
        except Exception as e:
            all_scores.extend([0.5] * len(batch))

    return dict(zip(valid_ids, all_scores))

## 8. Final Pipeline (3-Way Pure RRF)

In [ ]:
def final_pipeline(query, rerank_depth=RERANK_DEPTH, k_rrf=K_RRF):
    """
    Final Pipeline: 3-Way Pure RRF
    
    BM25 + RM3 + MonoT5-large (all weights = 1.0)
    
    Based on Cormack et al. (2009) RRF formula:
    RRFscore(d) = Σ 1/(k + rank(d))
    """
    # Stage 1: Get BM25 and RM3 rankings
    bm25_results, rm3_results = multi_method_retrieval(query)

    bm25_ranking = {docid: rank for rank, (docid, _) in enumerate(bm25_results, 1)}
    rm3_ranking = {docid: rank for rank, (docid, _) in enumerate(rm3_results, 1)}

    # Stage 2: Get MonoT5 ranking for top docs
    monot5_ranking = {}
    if USE_MONOT5 and monot5_model is not None:
        top_docids = [docid for docid, _ in bm25_results[:rerank_depth]]
        monot5_scores = rerank_monot5(query, top_docids)

        if monot5_scores and len(monot5_scores) > 0:
            sorted_by_monot5 = sorted(monot5_scores.items(), key=lambda x: x[1], reverse=True)
            monot5_ranking = {docid: rank for rank, (docid, _) in enumerate(sorted_by_monot5, 1)}

    # Stage 3: 3-Way Pure RRF Fusion (all weights = 1.0)
    all_docids = set(bm25_ranking.keys()) | set(rm3_ranking.keys())
    rrf_scores = {}

    for docid in all_docids:
        rrf_score = 0.0

        # BM25 contribution (weight = 1.0)
        if docid in bm25_ranking:
            rrf_score += 1.0 / (k_rrf + bm25_ranking[docid])

        # RM3 contribution (weight = 1.0)
        if docid in rm3_ranking:
            rrf_score += 1.0 / (k_rrf + rm3_ranking[docid])

        # MonoT5 contribution (weight = 1.0)
        if docid in monot5_ranking:
            rrf_score += 1.0 / (k_rrf + monot5_ranking[docid])

        rrf_scores[docid] = rrf_score

    # Sort by RRF score (descending)
    final_ranking = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # Build results
    results = []
    for rank, (docid, score) in enumerate(final_ranking[:1000], 1):
        results.append((str(docid), float(score), int(rank)))

    return results

## 9. Evaluate on Training Set

In [ ]:
print("="*70)
print("EVALUATING ON TRAINING SET")
print("="*70)
print("3-Way Pure RRF: BM25 + RM3 + MonoT5-large")
print(f"Parameters:")
print(f"  BM25: k1={BM25_K1}, b={BM25_B}")
print(f"  RM3: fb_docs={RM3_FB_DOCS}, fb_terms={RM3_FB_TERMS}, orig_weight={RM3_ORIG_WEIGHT}")
print(f"  k_rrf: {K_RRF}")
print("="*70)
print()

train_results = []
start_time = time.time()

for i, qid in enumerate(train_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(train_qids)}] Query {qid} ({query_type}): {query_text[:50]}...")

    try:
        results = final_pipeline(query_text)

        for docid, score, rank in results:
            if isinstance(score, np.ndarray):
                score = float(score.item())
            elif isinstance(score, list):
                score = float(score[0]) if score else 0.0
            else:
                score = float(score)

            train_results.append(ir_measures.ScoredDoc(
                query_id=str(qid),
                doc_id=str(docid),
                score=float(score)
            ))

        print(f"  Retrieved {len(results)} docs")
    except Exception as e:
        print(f"  Error: {str(e)[:100]}")
        continue

total_time = time.time() - start_time

# Convert qrels
qrels_list = []
for qid, doc_rels in qrels.items():
    for docid, rel in doc_rels.items():
        qrels_list.append(ir_measures.Qrel(
            query_id=str(qid),
            doc_id=str(docid),
            relevance=int(rel)
        ))

# Calculate metrics
print("\n" + "="*70)
print("COMPUTING METRICS...")
print("="*70)

metrics = [MAP, nDCG@10, nDCG@20, P@10, P@20]
results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, train_results)

print("\n" + "="*70)
print("TRAINING SET RESULTS (50 queries)")
print("="*70)
print("\nYour Scores:")
print(f"  MAP:      {results_metrics[MAP]:.4f}  <- Main metric")
print(f"  nDCG@10:  {results_metrics[nDCG@10]:.4f}")
print(f"  nDCG@20:  {results_metrics[nDCG@20]:.4f}")
print(f"  P@10:     {results_metrics[P@10]:.4f}")
print(f"  P@20:     {results_metrics[P@20]:.4f}")

print(f"\nTime: {total_time:.1f}s ({total_time/len(train_qids):.1f}s per query)")

print("\n" + "="*70)
if results_metrics[MAP] >= 0.30:
    print("SUCCESS! MAP >= 0.30")
print("="*70)

## 10. Generate Test Results

In [ ]:
print("="*70)
print("GENERATING TEST RESULTS")
print("="*70)
print(f"Configuration:")
print(f"  3-Way Pure RRF: BM25 + RM3 + MonoT5-large")
print(f"  BM25: k1={BM25_K1}, b={BM25_B}")
print(f"  RM3: fb_docs={RM3_FB_DOCS}, fb_terms={RM3_FB_TERMS}, orig_weight={RM3_ORIG_WEIGHT}")
print(f"  k_rrf: {K_RRF}")
print(f"  Rerank depth: {RERANK_DEPTH}")
print(f"  Total test queries: {len(test_qids)}")
print(f"\nExpected MAP: ~{results_metrics[MAP]:.4f} (from training)")
print("="*70)
print()

test_results = []
start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(test_qids)}] Query {qid} ({query_type}): {query_text[:70]}...")

    query_start = time.time()
    results = final_pipeline(query_text)
    query_time = time.time() - query_start

    for docid, score, rank in results:
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)

        test_results.append({
            'qid': str(qid),
            'docid': str(docid),
            'rank': int(rank),
            'score': float(score)
        })

    print(f"  Retrieved {len(results)} docs in {query_time:.1f}s")

    if i % 20 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time

        print()
        print("-"*70)
        print(f"PROGRESS: {i}/{len(test_qids)} ({i/len(test_qids)*100:.1f}%)")
        print(f"  Elapsed: {elapsed/60:.1f} min, Remaining: ~{remaining/60:.1f} min")
        print(f"  Avg: {avg_time:.1f}s/query")
        if torch.cuda.is_available():
            print(f"  GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print("-"*70)
        print()

total_time = time.time() - start_time

print()
print("="*70)
print("GENERATION COMPLETE!")
print("="*70)
print(f"  Queries: {len(test_qids)}")
print(f"  Rankings: {len(test_results):,}")
print(f"  Total time: {total_time/60:.1f} min")
print(f"  Avg: {total_time/len(test_qids):.1f}s/query")
print("="*70)

## 10. Chunked Test Generation (Alternative)

**Use this instead of cell 10 if you want to process in chunks.**

Benefits:
- Saves progress after each chunk
- Resume if Colab disconnects
- ~1.5-2 hours per chunk instead of 7 hours total

### Instructions:
1. Set `START_CHUNK = 0` and run
2. After chunk completes, set `START_CHUNK = 1` and run again
3. Continue until all chunks are done
4. Run the "Merge Chunks" cell to combine results

In [ ]:
# ============================================================================
# CHUNKED TEST GENERATION (with auto-save progress)
# ============================================================================

import time
import os
import numpy as np
import json as json_module

# ============================================
# CONFIGURATION
# ============================================
CHUNK_SIZE = 50       # Queries per chunk (~1.5-2 hours each)
START_CHUNK = 0       # Which chunk to start from (0, 1, 2, 3...)
SAVE_EVERY = 10       # Save progress every N queries

# OUTPUT_DIR: Use current working directory (where notebook runs)
# In Colab with Google Drive, this is usually your Drive folder
OUTPUT_DIR = os.getcwd()
print(f"📁 Output directory: {OUTPUT_DIR}")
# ============================================

# Calculate chunks
total_queries = len(test_qids)
num_chunks = (total_queries + CHUNK_SIZE - 1) // CHUNK_SIZE

print("="*70)
print("📦 CHUNKED TEST GENERATION (with auto-save)")
print("="*70)
print(f"Total test queries: {total_queries}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Total chunks: {num_chunks}")
print(f"Starting from chunk: {START_CHUNK}")
print(f"Auto-save every: {SAVE_EVERY} queries")
print(f"Saving to: {OUTPUT_DIR}")
print("="*70)

# Process requested chunk
for chunk_idx in range(START_CHUNK, num_chunks):
    start_idx = chunk_idx * CHUNK_SIZE
    end_idx = min(start_idx + CHUNK_SIZE, total_queries)
    chunk_qids = test_qids[start_idx:end_idx]
    
    chunk_file = os.path.join(OUTPUT_DIR, f"run_chunk_{chunk_idx:02d}.res")
    progress_file = os.path.join(OUTPUT_DIR, f"run_chunk_{chunk_idx:02d}_progress.json")
    
    # Check if chunk already complete
    if os.path.exists(chunk_file):
        print(f"\n⏭️  Chunk {chunk_idx} already complete ({chunk_file}), skipping...")
        continue
    
    # Check for partial progress
    chunk_results = []
    start_query_idx = 0
    
    if os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            progress = json_module.load(f)
            chunk_results = progress.get('results', [])
            start_query_idx = progress.get('completed', 0)
            print(f"\n🔄 Resuming chunk {chunk_idx} from query {start_query_idx + 1}")
    
    print(f"\n{'='*70}")
    print(f"📦 PROCESSING CHUNK {chunk_idx}/{num_chunks-1}")
    print(f"   Queries {start_idx+1} to {end_idx} ({len(chunk_qids)} queries)")
    if start_query_idx > 0:
        print(f"   Resuming from query {start_query_idx + 1}")
    print(f"{'='*70}\n")
    
    chunk_start_time = time.time()
    
    for i, qid in enumerate(chunk_qids):
        # Skip already processed queries
        if i < start_query_idx:
            continue
            
        query_text = queries[qid]
        query_type = classify_query(query_text)
        
        print(f"[{i+1}/{len(chunk_qids)}] Query {qid} ({query_type}): {query_text[:50]}...")
        
        query_start = time.time()
        
        try:
            results = final_pipeline(query_text)
            
            for docid, score, rank in results:
                if isinstance(score, (list, np.ndarray)):
                    score = float(score[0]) if len(score) > 0 else 0.0
                else:
                    score = float(score)
                
                chunk_results.append({
                    'qid': str(qid),
                    'docid': str(docid),
                    'rank': int(rank),
                    'score': float(score)
                })
            
            query_time = time.time() - query_start
            print(f"  ✓ Retrieved {len(results)} docs in {query_time:.1f}s")
            
        except Exception as e:
            print(f"  ✗ Error: {str(e)[:100]}")
            continue
        
        # Auto-save progress every N queries
        if (i + 1) % SAVE_EVERY == 0:
            with open(progress_file, 'w') as f:
                json_module.dump({'completed': i + 1, 'results': chunk_results}, f)
            elapsed = time.time() - chunk_start_time
            avg_time = elapsed / (i + 1 - start_query_idx) if (i + 1 - start_query_idx) > 0 else 0
            remaining = (len(chunk_qids) - i - 1) * avg_time
            print(f"\n  💾 Progress saved ({i+1}/{len(chunk_qids)}) | Remaining: ~{remaining/60:.1f} min\n")
    
    # Save final chunk file
    with open(chunk_file, 'w') as f:
        for r in chunk_results:
            f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {r['score']:.6f} run3_chunk{chunk_idx:02d}\n")
    
    # Remove progress file
    if os.path.exists(progress_file):
        os.remove(progress_file)
    
    chunk_time = time.time() - chunk_start_time
    print(f"\n{'='*70}")
    print(f"✓ CHUNK {chunk_idx} COMPLETE!")
    print(f"  Saved to: {chunk_file}")
    print(f"  Queries: {len(chunk_qids)}")
    print(f"  Results: {len(chunk_results):,}")
    print(f"  Time: {chunk_time/60:.1f} minutes")
    print(f"{'='*70}")
    
    # Stop after one chunk
    print(f"\n🔔 Chunk {chunk_idx} saved!")
    print(f"   To continue: Set START_CHUNK = {chunk_idx + 1} and re-run this cell")
    break

print("\n" + "="*70)
print("📝 NEXT STEPS:")
print("="*70)
print(f"1. Chunk files saved to: {OUTPUT_DIR}")
print("2. To process next chunk: Increase START_CHUNK and re-run")
print("3. When all chunks done, run the MERGE cell below")
print("="*70)



## 10b. Merge Chunks

Run this after all chunks are complete to merge them into the final run file.

In [ ]:
# ============================================================================
# MERGE CHUNK FILES INTO FINAL RUN FILE
# ============================================================================

import os
import glob

# Use current working directory (same as where chunks were saved)
OUTPUT_DIR = os.getcwd()
FINAL_FILE = "run_3_final.res"

print("="*70)
print("🔗 MERGING CHUNK FILES")
print("="*70)
print(f"Looking in: {OUTPUT_DIR}")

# Find chunk files
chunk_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "run_chunk_*.res")))

print(f"Found {len(chunk_files)} chunk files:")
for f in chunk_files:
    print(f"  • {os.path.basename(f)}")

if len(chunk_files) == 0:
    print("\n⚠️  No chunk files found!")
    print("\nDebug info:")
    print(f"  Current directory: {os.getcwd()}")
    print(f"  Files in directory:")
    for f in os.listdir(OUTPUT_DIR)[:20]:
        print(f"    {f}")
else:
    # Merge
    all_results = []
    for chunk_file in chunk_files:
        with open(chunk_file, 'r') as f:
            lines = f.readlines()
            all_results.extend(lines)
            print(f"  Loaded {len(lines):,} lines from {os.path.basename(chunk_file)}")

    # Save merged file
    output_path = os.path.join(OUTPUT_DIR, FINAL_FILE)
    with open(output_path, 'w') as f:
        f.writelines(all_results)

    print(f"\n{'='*70}")
    print(f"✓ MERGED SUCCESSFULLY!")
    print(f"{'='*70}")
    print(f"  Output file: {output_path}")
    print(f"  Total lines: {len(all_results):,}")
    
    # Count unique queries
    query_ids = set(line.split()[0] for line in all_results if line.strip())
    print(f"  Unique queries: {len(query_ids)}")
    
    print(f"\n  First 5 lines:")
    for line in all_results[:5]:
        print(f"    {line.strip()}")
    print("="*70)



## 11. Save Results

In [ ]:
output_file = 'run_3_final.res'

with open(output_file, 'w') as f:
    for r in test_results:
        score = r['score']
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {score:.6f} run3_final\n")

print(f"Saved to {output_file}")
print("\nFirst 5 lines:")
with open(output_file, 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

print("\n" + "="*70)
print("FINAL SUBMISSION READY!")
print("="*70)
print("\nFinal Configuration:")
print(f"  BM25: k1={BM25_K1}, b={BM25_B}")
print(f"  RM3: fb_docs={RM3_FB_DOCS}, fb_terms={RM3_FB_TERMS}, original_query_weight={RM3_ORIG_WEIGHT}")
print(f"  k_rrf: {K_RRF}")
print(f"  rerank_depth: {RERANK_DEPTH}")
print(f"  Reranker: MonoT5-large")
print(f"  3-Way Pure RRF: BM25 + RM3 + MonoT5-large")
print(f"\nTraining MAP: {results_metrics[MAP]:.4f}")
print(f"Output file: {output_file}")
print("="*70)

## 12. Evaluate Final Results

Evaluate your run file against the full ROBUST04 qrels.

**Requirements:**
- `qrels.robust2004.txt` - Full ROBUST04 relevance judgments
- `run_3_final.res` - Your merged run file

In [ ]:
# ============================================================================
# EVALUATE FINAL RESULTS
# ============================================================================

from collections import defaultdict
import os

def load_qrels_full(qrel_file):
    """Load qrels file (format: qid iter docid rel)"""
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, _, docid, rel = parts[0], parts[1], parts[2], int(parts[3])
                qrels[qid][docid] = rel
    return qrels

def load_run(run_file):
    """Load run file (format: qid Q0 docid rank score run_name)"""
    runs = defaultdict(list)
    with open(run_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                qid, _, docid, rank, score = parts[0], parts[1], parts[2], int(parts[3]), float(parts[4])
                runs[qid].append((docid, rank, score))
    
    # Sort by rank
    for qid in runs:
        runs[qid].sort(key=lambda x: x[1])
    
    return runs

def compute_ap(ranked_docs, qrel_docs):
    """Compute Average Precision for a single query"""
    if not qrel_docs:
        return 0.0
    
    num_relevant = sum(1 for rel in qrel_docs.values() if rel > 0)
    if num_relevant == 0:
        return 0.0
    
    relevant_count = 0
    precision_sum = 0.0
    
    for rank, (docid, _, _) in enumerate(ranked_docs, 1):
        if docid in qrel_docs and qrel_docs[docid] > 0:
            relevant_count += 1
            precision_at_rank = relevant_count / rank
            precision_sum += precision_at_rank
    
    return precision_sum / num_relevant

def compute_precision_at_k(ranked_docs, qrel_docs, k=10):
    """Compute Precision at k"""
    top_k = ranked_docs[:k]
    relevant_in_top_k = sum(1 for docid, _, _ in top_k 
                            if docid in qrel_docs and qrel_docs[docid] > 0)
    return relevant_in_top_k / k

# ============================================
# CONFIGURATION
# ============================================
QRELS_FILE = "qrels.robust2004.txt"  # Full ROBUST04 qrels
RUN_FILE = "run_3_final.res"         # Your merged run file
# ============================================

print("="*70)
print("📊 EVALUATING FINAL RESULTS")
print("="*70)
print(f"Run file: {RUN_FILE}")
print(f"Qrels file: {QRELS_FILE}")
print("="*70)

# Check files exist
if not os.path.exists(QRELS_FILE):
    print(f"\n⚠️  Qrels file not found: {QRELS_FILE}")
    print("   Upload qrels.robust2004.txt to your working directory")
elif not os.path.exists(RUN_FILE):
    print(f"\n⚠️  Run file not found: {RUN_FILE}")
    print("   Run the merge cell first to create the run file")
else:
    # Load files
    print("\nLoading files...")
    qrels_full = load_qrels_full(QRELS_FILE)
    runs = load_run(RUN_FILE)
    
    print(f"  Qrels: {len(qrels_full)} queries")
    print(f"  Run: {len(runs)} queries")
    
    # Compute metrics
    print("\nComputing metrics...")
    
    aps = []
    p10s = []
    p20s = []
    
    for qid in sorted(qrels_full.keys()):
        if qid in runs:
            ap = compute_ap(runs[qid], qrels_full[qid])
            p10 = compute_precision_at_k(runs[qid], qrels_full[qid], k=10)
            p20 = compute_precision_at_k(runs[qid], qrels_full[qid], k=20)
        else:
            ap, p10, p20 = 0.0, 0.0, 0.0
        
        aps.append(ap)
        p10s.append(p10)
        p20s.append(p20)
    
    map_score = sum(aps) / len(aps) if aps else 0.0
    avg_p10 = sum(p10s) / len(p10s) if p10s else 0.0
    avg_p20 = sum(p20s) / len(p20s) if p20s else 0.0
    
    print("\n" + "="*70)
    print("📊 RESULTS")
    print("="*70)
    print(f"  MAP:       {map_score:.4f}  ← Main metric")
    print(f"  P@10:      {avg_p10:.4f}")
    print(f"  P@20:      {avg_p20:.4f}")
    print(f"  Queries:   {len(aps)}")
    print("="*70)
    
    # Performance assessment
    print("\n📈 Performance Assessment:")
    if map_score >= 0.35:
        print("  🌟 EXCELLENT! Top-tier performance!")
    elif map_score >= 0.30:
        print("  ✓ GREAT! Strong performance!")
    elif map_score >= 0.28:
        print("  ✓ GOOD! Solid performance!")
    elif map_score >= 0.25:
        print("  ⚠️  OK - room for improvement")
    else:
        print("  ⚠️  Below expected - check configuration")
    print("="*70)
